In [ ]:
!pip install datasets faiss-cpu==1.7.4 chromadb==0.4.22 sentence-transformers==2.3.1

In [ ]:
from datasets import load_dataset

qna_dataset = load_dataset("sadeem-ai/arabic-qna")

In [ ]:
qna_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'text', 'source', 'question', 'answer', 'has_answer'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['title', 'text', 'source', 'question', 'answer', 'has_answer'],
        num_rows: 1030
    })
})

In [ ]:
qna_dataset = qna_dataset.filter(lambda example: example["has_answer"] == True)

In [ ]:
doc_texts = qna_dataset["train"]["text"]

In [ ]:
len(doc_texts)

4037

In [ ]:
metadata = [
    {
        "source": rec["source"],
        "title": rec["title"]
    }
    for rec in qna_dataset["train"]
]

In [ ]:
len(metadata)

4037

In [ ]:
metadata[2110]

{'source': 'https://ar.wikipedia.org/wiki?curid=1305828',
 'title': 'حدسية أوبيرمان'}

In [ ]:
docs_ids = [
    str(i)
    for i in range( len(doc_texts) )
]

In [ ]:
len(docs_ids)

4037

## Text to Vectors

In [ ]:
from sentence_transformers import SentenceTransformer

model_id = "sentence-transformers/distiluse-base-multilingual-cased-v2"
dim = 512

# model_id = "asafaya/bert-large-arabic"
# dim = 1024

device = "cuda:0" # "cpu"

model = SentenceTransformer(model_id, device=device)

/usr/local/lib/python3.10/dist-packages/sentence_transformers/models/Dense.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(input_path, "pytorch_

In [ ]:
encoded_docs = model.encode(doc_texts, show_progress_bar=True)

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
encoded_docs

array([[-0.02226301,  0.01186228, -0.05172828, ..., -0.03353376,
        -0.05095804, -0.00841591],
       [-0.04920378,  0.00141093,  0.04245442, ...,  0.01974735,
         0.0204893 , -0.00664262],
       [ 0.04379221, -0.02765544,  0.01548383, ...,  0.01340353,
        -0.04520679,  0.04465359],
       ...,
       [-0.01370987, -0.05809394,  0.05020998, ..., -0.06363255,
        -0.04193227,  0.0467475 ],
       [-0.01150203, -0.01077436, -0.00982356, ..., -0.02248277,
         0.01633337,  0.01352999],
       [-0.00064259,  0.00782018, -0.04364197, ..., -0.02850981,
         0.05100543, -0.05257529]], dtype=float32)

In [ ]:
encoded_docs.shape

(4037, 512)

In [ ]:
doc_texts[10], encoded_docs[10]

('حاصل على الدكتوراه في تخصص النمذجة الرياضية للأنظمة الكيميائية الدقيقة من جامعة ليهاي الولايات المتحدة الأمريكية تخصص الهندسة الكيميائية عام 2005.',
 array([-0.00827502,  0.02703173, -0.04986278, -0.05056055, -0.00715024,
         0.01376725, -0.01454555, -0.02480281, -0.00047607, -0.01045043,
         0.03023715, -0.02332602, -0.01728399, -0.03127356,  0.01887567,
        -0.02433514, -0.00328176,  0.02666474,  0.03013173, -0.00949638,
        -0.05833135,  0.04672301, -0.05709618, -0.06757794,  0.01133903,
        -0.0081391 ,  0.01069322,  0.0249118 , -0.05644627,  0.00968402,
         0.02276987, -0.00397209,  0.00232532,  0.04947095,  0.03763633,
         0.01476778, -0.00333424,  0.02307385,  0.02232516, -0.02466128,
        -0.04907871,  0.0181725 , -0.03494672, -0.01811381,  0.05829423,
        -0.10454393,  0.02031245, -0.01067277,  0.02680671,  0.04748167,
         0.04696176,  0.02541604,  0.00758475, -0.03070021, -0.02502944,
        -0.00381934, -0.00362916, -0.00486979,

## Vector Databases

### ChromaDB

In [ ]:
# Building a DB in ChromaDB.
import chromadb

# Using PersistentClient to save anything in ChromDB To the Disk
chroma_client = chromadb.PersistentClient(path="./chromadb-ar-docs")

In [ ]:
collection_2 = chroma_client.create_collection(
    name="first_coll",
    metadata={"hnsw:space": "cosine"}
)

In [ ]:
collection_2.add(
    documents=doc_texts,
    embeddings=encoded_docs,
    metadatas=metadata,
    ids=docs_ids
)

In [ ]:
collection_3 = chroma_client.create_collection(
    name="yarab_coll",
    metadata={"hnsw:space": "cosine"}
)

In [ ]:
collection_3.add(
    documents=doc_texts,
    embeddings=encoded_docs,
    metadatas=metadata,
    ids=docs_ids
)

In [ ]:
## Search
question = "من صاحب شركة لونجمان بعد وفاة لونجمان الأول؟"
question_embed = model.encode(question)

results = collection_3.query(
    query_embeddings=question_embed.tolist(),
    n_results=3
)

In [ ]:
print(question_embed.shape)  # Should match `dim` (e.g., 512 or 1024)
print(question_embed[:5])  # Check for reasonable embedding values

(512,)
[-0.02375775  0.00564564 -0.01569494  0.02344877 -0.04091102]


In [ ]:
results

{'ids': [['2110', '347', '3478']],
 'distances': [[0.756523847579956, 0.7856284379959106, 0.7856284379959106]],
 'metadatas': [[{'source': 'https://ar.wikipedia.org/wiki?curid=1305828',
    'title': 'حدسية أوبيرمان'},
   {'source': 'https://ar.wikipedia.org/wiki?curid=948722',
    'title': 'بيركيركارا'},
   {'source': 'https://ar.wikipedia.org/wiki?curid=948722',
    'title': 'بيركيركارا'}]],
 'embeddings': None,
 'documents': [['في الرياضيات، حدسية أوبيرمان (مسماة هكذا نسبة إلى لودفيش أوبرمان)، تهم توزيع الأعداد الأولية. تنص هاته الحدسية على أنه بالنسبة لأي عدد x أكبر قطعا من 1، فإن يوجد عدد أولي واحد على الأقل محصور بين (x(x\xa0−\xa01 و \xa0x,',
   'هنالك شخصيتان بارزتان ينحدران من البلدة وهما إيدي فرينش آدامي وألفريد سانت اللذين شَغِلا منصب رئيس وزراء مالطا لعدد من السنوات. كما تُعدّ البلدة مسقط رأس رئيسها الأول أنتوني مامو.',
   'هنالك شخصيتان بارزتان ينحدران من البلدة وهما إيدي فرينش آدامي وألفريد سانت اللذين شَغِلا منصب رئيس وزراء مالطا لعدد من السنوات. كما تُعدّ البلدة مسقط رأس 

In [ ]:
question2 = "ما هو العامل المرتبط بزيادة خطر الاصابة بالمرض؟"
question_embed = model.encode(question2)

results_2 = collection_2.query(
    query_embeddings=question_embed.tolist(),
    n_results=3
)

In [ ]:
collection_2.count()

4037

In [ ]:
results_2

{'ids': [['321', '1560', '2560']],
 'distances': [[0.6987627148628235, 0.6987627148628235, 0.7444360256195068]],
 'metadatas': [[{'source': 'https://ar.wikipedia.org/wiki?curid=6845739',
    'title': 'أسباب الاضطراب الوسواسي القهري'},
   {'source': 'https://ar.wikipedia.org/wiki?curid=6845739',
    'title': 'أسباب الاضطراب الوسواسي القهري'},
   {'source': 'https://ar.wikipedia.org/wiki?curid=154148',
    'title': 'مرض خدش القطة'}]],
 'embeddings': None,
 'documents': [['قدمت الباحثتان هنريتا ليونارد وسوزان سويدو أدلة على وجود عوامل خطر مناعية عصبية ذات علاقة باضطراب الوسواس القهري، وذلك من خلال ورقة بحثية تحت عنوان «اختلالات المناعة الذاتية النفسية والعصبية المصاحبة لعدوى البكتيريا الكروية السبحية في الأطفال» (التي يُطلق عليها اختصارًا اسم باندز). يقترح الباحثون أن المناعة الذاتية المتولدة بعد العدوى من البكتيريا العقدية من الأسباب البيئية المحيطة المحتملة لظهور الوسواس القهري لدى الأطفال. إذ أظهرت مجموعة من الأطفال تفاقمًا في حدة أعراض الوسواس القهري بعد إصابتهم بعدوى البكتيريا العقدي

### FAISS

In [ ]:
import faiss
import numpy as np
from copy import deepcopy

In [ ]:
norm_encoded_docs = deepcopy(encoded_docs)
faiss.normalize_L2(norm_encoded_docs)

In [ ]:
faiss_index = faiss.IndexIDMap( faiss.IndexFlatIP(dim) )

faiss_index.add_with_ids( norm_encoded_docs, docs_ids )

In [ ]:
question = "ما السبب في صغر الأسنان بالمقارنة مع حجم الفكين؟"
question_embed = model.encode([question])

faiss.normalize_L2(question_embed)

results = faiss_index.search(question_embed, 3)

In [ ]:
print(results)

(array([[0.5805141 , 0.5805141 , 0.33630452]], dtype=float32), array([[1534,  397, 3407]]))


In [ ]:
doc_texts[1534], doc_texts[397], doc_texts[3407]

('جميع الأسنان ذات حجم طبيعي ولكنها تبدو صغيرة بسبب ضخامة الفكين. قد يكون الصغر النسبي المعمم نتيجة وراثة فك كبير من أحد الوالدين، وأسنان ذات حجم طبيعي من الآخر.',
 'جميع الأسنان ذات حجم طبيعي ولكنها تبدو صغيرة بسبب ضخامة الفكين. قد يكون الصغر النسبي المعمم نتيجة وراثة فك كبير من أحد الوالدين، وأسنان ذات حجم طبيعي من الآخر.',
 'أُخْدودُ لسان المِزْمار أو الوهدة هو انخفاض (أخدود) خلف جذر اللسان بين الطيات في الحلق. هذه المنخفضات تكون بمثابة "مصائد لتجميع البصاق"؛ حيث يتجمع اللعاب مؤقتا في الوهدات لمنع مُنكعس البلع من أن يبدأ.')

In [ ]:
## Save
import pickle

with open("./faiss-ar-docs/index.pickle", "wb") as handle:
    pickle.dump(faiss_index, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open("./faiss-ar-docs/data.pickle", "wb") as handle:
    pickle.dump({
        "data": doc_texts,
        "docs_ids": docs_ids,
        "metadata": metadata
    }, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
## Load
import pickle

with open("./faiss-ar-docs/index.pickle", "rb") as handle:
    loaded_faiss_index = pickle.load(handle)

with open("./faiss-ar-docs/data.pickle", "rb") as handle:
    loaded_faiss_data = pickle.load(handle)

## Let's Compare

### Retrieving Speed

In [ ]:
import time

**Dim: 512**

---------
`ChromaDB`: 4037 -
12.698187112

`FIASS`: 4037 -
2.043501231999997

---------
**Dim: 1024**

`ChromaDB`: 4037
15.938859471999997

`FIASS`: 4037
4.1828746040000055


In [ ]:
t0 = time.process_time()

for i in range(len(doc_questions)):

    ques = encoded_questions[i]

    results = collection.query(
        query_embeddings=ques.tolist(),
        n_results=3
    )

print("ChromaDB:", len(doc_questions))
print(time.process_time() - t0)

ChromaDB: 4037
13.255644789000002


In [ ]:
t0 = time.process_time()

for i in range(len(doc_questions)):

    ques = encoded_questions[i].reshape(1, dim)

    faiss.normalize_L2(ques)

    results = faiss_index.search(ques, 3)


print("FIASS:", len(doc_questions))
print(time.process_time() - t0)

FIASS: 4037
24.845981306


### Accuracy

`ChromaDB`

```
Model ID: sentence-transformers/distiluse-base-multilingual-cased-v2
----
Valid: 1201
Valid%: 0.2974981421847907
----
Similar: 864
Similar%: 0.21402031211295516
----
InValid: 1972
InValid%: 0.48848154570225416
----


Model ID: asafaya/bert-large-arabic
----
Valid: 586
Valid%: 0.14515729502105523
----
Similar: 427
Similar%: 0.10577161258360168
----
InValid: 3024
InValid%: 0.7490710923953431
----

```

`FAISS`

```
Model ID: sentence-transformers/distiluse-base-multilingual-cased-v2
----
Valid: 1374
Valid%: 0.3403517463462968
----
Similar: 947
Similar%: 0.23458013376269507
----
InValid: 1716
InValid%: 0.4250681198910082
----

Model ID: asafaya/bert-large-arabic
----
Valid: 703
Valid%: 0.17413921228635126
----
Similar: 518
Similar%: 0.12831310378994304
----
InValid: 2816
InValid%: 0.6975476839237057
----
```

In [ ]:
chroma_results = []

for i in range(len(doc_questions)):

    ques = encoded_questions[i]

    results = collection.query(
        query_embeddings=ques.tolist(),
        n_results=3
    )

    chroma_results.append(results)

In [ ]:
chroma_insights = {
    "valid": 0,
    "similar": 0,
    "invalid": 0
}

for i in range(len(doc_questions)):
    true_id = docs_ids[i]
    pred_id = chroma_results[i]["ids"][0][0]

    true_source = metadata[i]["source"]
    pred_source = metadata[int(pred_id)]["source"]

    if str(true_id) == str(pred_id):
        chroma_insights["valid"] += 1

    elif true_source == pred_source:
        chroma_insights["similar"] += 1

    else:
        chroma_insights["invalid"] += 1

chroma_insights["valid_percentage"] = chroma_insights["valid"]/len(doc_questions)
chroma_insights["similar_percentage"] = chroma_insights["similar"]/len(doc_questions)
chroma_insights["invalid_percentage"] = chroma_insights["invalid"]/len(doc_questions)

print("Model ID:", model_id)
print("----")
print("Valid:", chroma_insights["valid"])
print("Valid%:", chroma_insights["valid_percentage"])
print("----")
print("Similar:", chroma_insights["similar"])
print("Similar%:", chroma_insights["similar_percentage"])
print("----")
print("InValid:", chroma_insights["invalid"])
print("InValid%:", chroma_insights["invalid_percentage"])
print("----")

Model ID: sentence-transformers/distiluse-base-multilingual-cased-v2
----
Valid: 748
Valid%: 0.18528610354223432
----
Similar: 514
Similar%: 0.12732226901164231
----
InValid: 2775
InValid%: 0.6873916274461234
----


In [ ]:
faiss_results = []

for i in range(len(doc_questions)):

    ques = encoded_questions[i].reshape(1, dim)

    faiss.normalize_L2(ques)

    scores, ids = faiss_index.search(ques, 3)

    faiss_results.append({
        "scores": scores,
        "ids": ids
    })


In [ ]:
faiss_insights = {
    "valid": 0,
    "similar": 0,
    "invalid": 0
}

for i in range(len(doc_questions)):
    true_id = docs_ids[i]
    pred_id = faiss_results[i]["ids"][0][0]

    true_source = metadata[i]["source"]
    pred_source = metadata[int(pred_id)]["source"]

    if str(true_id) == str(pred_id):
        faiss_insights["valid"] += 1

    elif true_source == pred_source:
        faiss_insights["similar"] += 1

    else:
        faiss_insights["invalid"] += 1


faiss_insights["valid_percentage"] = faiss_insights["valid"]/len(doc_questions)
faiss_insights["similar_percentage"] = faiss_insights["similar"]/len(doc_questions)
faiss_insights["invalid_percentage"] = faiss_insights["invalid"]/len(doc_questions)


print("Model ID:", model_id)
print("----")
print("Valid:", faiss_insights["valid"])
print("Valid%:", faiss_insights["valid_percentage"])
print("----")
print("Similar:", faiss_insights["similar"])
print("Similar%:", faiss_insights["similar_percentage"])
print("----")
print("InValid:", faiss_insights["invalid"])
print("InValid%:", faiss_insights["invalid_percentage"])
print("----")

Model ID: sentence-transformers/distiluse-base-multilingual-cased-v2
----
Valid: 907
Valid%: 0.2246717859796879
----
Similar: 638
Similar%: 0.15803814713896458
----
InValid: 2492
InValid%: 0.6172900668813476
----
